In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from google.colab import drive

# connection to drive
drive.mount('/content/drive')

# device definition
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader, random_split

class SpotifyMultiModalDataset(Dataset):
    """
    A PyTorch Dataset class for loading and preprocessing multimodal Spotify data.
    It combines audio Mel-spectrograms, textual lyrics embeddings, and numerical
    features (release year) to predict track popularity.

    The dataset is automatically balanced during initialization by oversampling
    the minority classes (hits and flops) to match the size of the mid-tier class.
    """
    def __init__(self, csv_path, spectro_dir, embeddings_path, ids_path):
        """
        Initializes the dataset by loading the metadata, balancing the classes,
        and mapping the available spectrograms and text embeddings.

        Args:
            csv_path (str): Path to the CSV file containing track metadata and popularity scores.
            spectro_dir (str): Directory containing the saved audio Mel-spectrograms (.npy files).
            embeddings_path (str): Path to the .npy file containing Sentence-BERT text embeddings.
            ids_path (str): Path to the .npy file containing track IDs corresponding to the embeddings.
        """
        print("Loading data and creating balanced dataset...")
        raw_df = pd.read_csv(csv_path)

        hits = raw_df[raw_df['popularity'] >= 65]
        mids = raw_df[(raw_df['popularity'] > 35) & (raw_df['popularity'] < 65)]
        flops = raw_df[raw_df['popularity'] <= 35]

        target_size = len(mids)

        hits_balanced = hits.sample(target_size, replace=True)
        flops_balanced = flops.sample(target_size, replace=True)

        self.df = pd.concat([flops_balanced, mids, hits_balanced]).reset_index(drop=True)

        self.embeddings = np.load(embeddings_path)
        emb_track_ids = np.load(ids_path)
        self.tid_to_emb_idx = {tid: i for i, tid in enumerate(emb_track_ids)}
        self.spectro_dir = spectro_dir

        self.valid_data = []
        for idx, row in self.df.iterrows():
            tid = row['track_id']
            spectro_file = os.path.join(self.spectro_dir, f"{tid}.npy")
            if os.path.exists(spectro_file) and tid in self.tid_to_emb_idx:
                self.valid_data.append({
                    'csv_row_idx': idx,
                    'track_id': tid,
                    'spectro_path': spectro_file,
                    'emb_idx': self.tid_to_emb_idx[tid]
                })
        print(f"Dataset ready! Balanced size: {len(self.valid_data)} samples.")

    def __len__(self):
        """
        Returns the total number of valid samples available in the dataset.

        Returns:
            int: Number of valid data samples.
        """
        return len(self.valid_data)

    def __getitem__(self, idx):
        """
        Retrieves a single multimodal sample from the dataset at the specified index.
        It pads or truncates the audio spectrogram to a fixed width of 1200 frames.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing:
                - spectro_tensor (torch.Tensor): The preprocessed Mel-spectrogram (1, n_mels, 1200).
                - text_vec (torch.Tensor): The text embedding vector representing the lyrics.
                - year_tensor (torch.Tensor): The normalized release year.
                - target (torch.Tensor): The normalized popularity score (between 0 and 1).
        """
        item_info = self.valid_data[idx]

        # Audio
        spectro = np.load(item_info['spectro_path'])
        target_width = 1200
        if spectro.shape[1] > target_width:
            spectro = spectro[:, :target_width]
        else:
            padding = target_width - spectro.shape[1]
            spectro = np.pad(spectro, ((0,0), (0, padding)), mode='constant')
        spectro_tensor = torch.FloatTensor(spectro).unsqueeze(0)

        # Text
        text_vec = torch.FloatTensor(self.embeddings[item_info['emb_idx']])

        # Year
        row = self.df.iloc[item_info['csv_row_idx']]
        year = row.get('year', 2019)
        normalized_year = (float(year) - 1950) / 75.0
        year_tensor = torch.FloatTensor([normalized_year])

        # Popularity (Target is between 0 and 1)
        target = torch.FloatTensor([row['popularity'] / 100.0])

        return spectro_tensor, text_vec, year_tensor, target

# Path definition and loaders definition
PATH_CSV = "/content/drive/MyDrive/Spotify_MASTER_Dataset.csv"
DIR_SPECTROS = "/content/drive/MyDrive/Spotify_Project_Data/mel_spectrograms"
PATH_EMB = "/content/drive/MyDrive/Spotify_Project_Data/lyrics_embeddings.npy"
PATH_IDS = "/content/drive/MyDrive/Spotify_Project_Data/embeddings_track_ids.npy"

full_dataset = SpotifyMultiModalDataset(PATH_CSV, DIR_SPECTROS, PATH_EMB, PATH_IDS)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

Loading data and creating balanced dataset...


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class HitPredictor(nn.Module):
    """
    A multimodal deep learning architecture designed to predict the commercial
    success (hit potential) of a song.

    The model integrates three distinct data modalities:
    1. Audio: Processes Mel-spectrograms (as 1-channel images) using a modified ResNet50 backbone.
    2. Text: Accepts pre-computed dense embeddings (e.g., from Sentence-BERT) representing the lyrics.
    3. Metadata: Incorporates the normalized release year.

    The extracted features from all modalities are concatenated and passed through
    a Multi-Layer Perceptron (MLP) head to output a final probability score between 0 and 1.
    """
    def __init__(self, text_emb_dim=384, audio_out_dim=512):
        """
        Initializes the HitPredictor network architecture.

        Args:
            text_emb_dim (int): The dimensionality of the input text embeddings (default is 384 for all-MiniLM-L6-v2).
            audio_out_dim (int): The target dimensionality for the ResNet50 audio feature extraction layer.
        """
        super(HitPredictor, self).__init__()

        # ResNet Audio Branch
        self.audio_branch = models.resnet50(weights=None)
        # Modify the first convolutional layer to accept 1-channel input (spectrograms) instead of 3-channel RGB.
        self.audio_branch.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        # Modify the final fully connected layer to output the desired audio dimension.
        self.audio_branch.fc = nn.Linear(self.audio_branch.fc.in_features, audio_out_dim)

        # MLP Head
        combined_dim = audio_out_dim + text_emb_dim + 1

        self.mlp = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, spectro, text_vec, year):
        """
        Defines the forward pass for predicting the hit score.

        Args:
            spectro (torch.Tensor): A batch of input Mel-spectrograms of shape (batch_size, 1, height, width).
            text_vec (torch.Tensor): A batch of input lyrics embeddings of shape (batch_size, text_emb_dim).
            year (torch.Tensor): A batch of normalized release years of shape (batch_size, 1) or (batch_size,).

        Returns:
            torch.Tensor: A batch of predicted hit probability scores of shape (batch_size, 1),
                          with values ranging from 0.0 to 1.0.
        """
        audio_vec = self.audio_branch(spectro)
        year = year.view(-1, 1)
        combined = torch.cat((audio_vec, text_vec, year), dim=1)
        hit_score = self.mlp(combined)
        return hit_score

model = HitPredictor(text_emb_dim=384, audio_out_dim=512).to(device)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

class WeightedMSELoss(nn.Module):
    """
    Custom Mean Squared Error (MSE) loss function designed to penalize
    predictions more heavily when the ground truth target values are edge cases
    (e.g., extreme hits or extreme flops).
    """
    def __init__(self):
        """
        Initializes the WeightedMSELoss module.
        """
        super().__init__()

    def forward(self, pred, target):
        """
        Computes the weighted mean squared error between predictions and targets.

        The weight increases linearly as the target value moves away from the
        median (0.5). For example, a target of 0.90 will receive a much higher
        weight (3.5) compared to a target of 0.50 (weight 1.0).

        Args:
            pred (torch.Tensor): The predicted values from the model.
            target (torch.Tensor): The ground truth target values.

        Returns:
            torch.Tensor: The computed scalar loss value.
        """
        distance_from_center = torch.abs(target - 0.5)
        weights = 1.0 + 5.0 * distance_from_center

        squared_errors = (pred - target) ** 2
        weighted_errors = weights * squared_errors
        return weighted_errors.mean()

# Initialize the custom weighted loss function
criterion = WeightedMSELoss()

# Initialize the Adam optimizer with a slightly reduced learning rate for more stable convergence
optimizer = optim.Adam(model.parameters(), lr=0.0005)

num_epochs = 10
best_val_loss = float('inf')

print("🚀 Starting Weighted Training Loop...")

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for spectro, text_vec, year, target_score in train_bar:

        spectro = spectro.to(device)
        text_vec = text_vec.to(device)
        year = year.to(device)
        target_score = target_score.to(device)

        optimizer.zero_grad()
        predictions = model(spectro, text_vec, year)

        loss = criterion(predictions.view(-1), target_score.view(-1))
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = running_loss / len(train_loader)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for spectro, text_vec, year, target_score in val_bar:
            spectro = spectro.to(device)
            text_vec = text_vec.to(device)
            year = year.to(device)
            target_score = target_score.to(device)

            predictions = model(spectro, text_vec, year)
            loss = criterion(predictions.view(-1), target_score.view(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"📊 Epoch [{epoch+1}/{num_epochs}] Summary: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Model Checkpointing ---
    # Save the model state if the validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/Spotify_Project_Data/best_hit_predictor_sharp.pth")
        print(f"⭐ New SHARP model saved! (Val Loss: {best_val_loss:.4f})")

print("🎉 Training Complete!")